Preprocessing and Feature Engineering — Group 2026-DS-77

Responsible member: IT24103676

Input: The existing development split created by the verification and EDA notebook.

Objective: Develop justified, reusable preprocessing pipelines for churn classification.

The raw dataset and saved split files remain unchanged. The reserved test set is excluded from this notebook. Any preparation that learns from data—including imputation, encoding and scaling—will be fitted within training folds during model development.
Decisions will draw on development-data evidence and be recorded in the Preprocessing and Feature Log.

In [1]:
#Mount Drive and locate the existing split

from pathlib import Path
from google.colab import drive

# Connect Colab to the shared project files in Google Drive.
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ML 2026-DS-77 Telco Customer Churn"
)

SPLIT_DIR = PROJECT_DIR / "data" / "splits"

# Stop clearly if the expected project folder cannot be found.
if not SPLIT_DIR.is_dir():
    raise FileNotFoundError(
        f"Split folder not found: {SPLIT_DIR}\n"
        "Check the project folder name and Drive location."
    )

# List filenames only. This does not read the reserved test data.
print("Available split files:")
for file_path in sorted(SPLIT_DIR.iterdir()):
    if file_path.is_file():
        print(file_path.name)

Mounted at /content/drive
Available split files:
development.csv
development_cv_manifest.csv
development_cv_settings.json
split_manifest.csv
split_record.json
test.csv


In [2]:
#Load and verify the development data

import pandas as pd

# Read only the existing development split.
# Keep customer IDs as text for traceability.
DEV_PATH = SPLIT_DIR / "development.csv"
dev = pd.read_csv(DEV_PATH, dtype={"customerID": "string"})

# These expectations come from Member 2's completed handover.
assert dev.shape == (5634, 21), (
    f"Unexpected development shape: {dev.shape}"
)
assert dev["customerID"].notna().all(), "Missing customer IDs found."
assert dev["customerID"].is_unique, "Duplicate customer IDs found."
assert dev["Churn"].notna().all(), "Missing target values found."
assert set(dev["Churn"].unique()) == {"No", "Yes"}, (
    "Unexpected Churn labels found."
)

expected_counts = {"No": 4139, "Yes": 1495}
actual_counts = dev["Churn"].value_counts().to_dict()

assert actual_counts == expected_counts, (
    f"Target counts differ from the handover: {actual_counts}"
)

print("Development checks passed.")
print("Shape:", dev.shape)
display(dev["Churn"].value_counts().rename("Customers").to_frame())

#These checks confirm the expected structure and counts. They do not replace the source-and-manifest integrity checks already performed in the first notebook.


Development checks passed.
Shape: (5634, 21)


,Customers
Churn,
No,4139
Yes,1495


In [3]:
#Convert TotalCharges safely
#The CSV may store this column as text because it contains blank entries. We need numerical values while preserving those blanks as missing.

# Work on an independent in-memory copy.
# This does not change development.csv.
prepared = dev.copy()

# Standardise surrounding whitespace before numeric conversion.
total_text = prepared["TotalCharges"].astype("string").str.strip()
blank_or_missing = total_text.isna() | total_text.eq("").fillna(False)

total_numeric = pd.to_numeric(
    total_text.mask(blank_or_missing),
    errors="coerce"
)

# Detect unexpected text rather than silently treating it as a known blank.
unexpected_text = total_numeric.isna() & ~blank_or_missing

assert not unexpected_text.any(), (
    "Unexpected non-numeric TotalCharges values found. "
    "Inspect these before continuing."
)

prepared["TotalCharges"] = total_numeric.astype("float64")

assert prepared["TotalCharges"].isna().sum() == 8, (
    "Missing TotalCharges count differs from the EDA handover."
)

print(
    "Missing TotalCharges:",
    prepared["TotalCharges"].isna().sum()
)

# Inspect the affected development records to confirm the EDA finding.
display(
    prepared.loc[
        prepared["TotalCharges"].isna(),
        ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
    ]
)

#Why this is safe before cross-validation: converting a numerical string into a number follows a fixed rule. It does not learn a median, average or other parameter from the data. We have not filled the missing values yet.

Missing TotalCharges: 8


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
403,4472-LVYGI,0,52.55,NaN,No
883,4367-NUYAO,0,25.75,NaN,No
1093,1371-DWPAZ,0,56.05,NaN,No
2691,7644-OMVMY,0,19.85,NaN,No
3069,3213-VVOLG,0,25.35,NaN,No
3502,2520-SGTTA,0,20.00,NaN,No
5332,4075-WKNIU,0,73.35,NaN,No
5400,2775-SEFEE,0,61.90,NaN,No


In [4]:
#Separate predictors, target and identifiers

# Retain identifiers separately for joining predictions to customers later.
customer_ids = prepared["customerID"].copy()

# Positive class: recorded churn.
y = prepared["Churn"].map({"No": 0, "Yes": 1}).astype("int64")

# Exclude the outcome and customer identifier from model inputs.
X = prepared.drop(columns=["customerID", "Churn"]).copy()

assert X.shape == (5634, 19)
assert X.index.equals(y.index)
assert X.index.equals(customer_ids.index)

print("Predictor shape:", X.shape)
print("Target encoding: No = 0, Yes = 1")
print("Customer IDs retained separately.")

Predictor shape: (5634, 19)
Target encoding: No = 0, Yes = 1
Customer IDs retained separately.


Including Churn in X would reveal the answer to the model. customerID identifies records but is excluded as a predictor.

Checks passed and match Member 2’s handover. We now have 19 candidate predictors, the correctly encoded target, and customer IDs stored separately.

The eight missing TotalCharges values all belong to customers with tenure zero. This suggests a connection with newly joined customers, but it does not establish that their true total charge is zero.

Missing TotalCharges — initial preprocessing decision

Eight development records have missing TotalCharges. All eight have tenure zero. We will retain these records and initially use median imputation, fitted only on the training portion of each cross-validation fold.

We selected this approach because it preserves valid records, provides a reproducible treatment for missing inputs, and is less sensitive to extreme charge amounts than mean imputation.

We did not assume that blank values represent zero because the reviewed documentation does not establish this.

The limitation is that the overall training median may not represent the actual accumulated charge of a tenure-zero customer. This is an initial modelling treatment, not a claim that the missing values have been recovered. Its predictive suitability remains to be evaluated.

In [5]:
# Do not use this approach before cross-validation:
# X["TotalCharges"] = X["TotalCharges"].fillna(X["TotalCharges"].median())
#That would use all development records to calculate the replacement value. Instead, the pipeline will calculate it separately from the training portion of each fold.


In [6]:
#Define the feature groups
# Numerical measurements: duration and charge amounts.
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

# Categorical attributes.
# SeniorCitizen is a category indicator even though it is stored as 0/1.
categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

# Check that every predictor appears exactly once.
assigned_features = numeric_features + categorical_features

assert len(assigned_features) == len(set(assigned_features)), (
    "A feature appears in more than one group."
)
assert set(assigned_features) == set(X.columns), (
    "Feature groups do not match the predictor columns."
)

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# Confirm missingness before defining the pipeline.
# Blank strings are counted separately because they are not always read as NaN.
missing_check = pd.DataFrame({
    "Missing values": X.isna().sum(),
    "Blank strings": [
        X[column]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
        .sum()
        for column in X.columns
    ],
})

display(missing_check)

Numerical features: 3
Categorical features: 16


,Missing values,Blank strings
gender,0,0
SeniorCitizen,0,0
Partner,0,0
Dependents,0,0
tenure,0,0
PhoneService,0,0
MultipleLines,0,0
InternetService,0,0
OnlineSecurity,0,0
OnlineBackup,0,0


We’ll define the preprocessing pipelines. Defining them sets the instructions; it does not yet calculate replacement values, learn categories or scale the data.

1. Understand the two preparation routes
Because your models are not yet selected, we will provide two versions of the same preparation:

| Feature group | Scaled version | Unscaled version |
|---|---|---|
| Three numerical features | Median imputation → standard scaling | Median imputation |
| Sixteen categorical features | One-hot encoding | One-hot encoding |

This avoids imposing numerical spacing between categories. We will also preserve "No internet service" and "No phone service" as meaningful categories.

We use handle_unknown="ignore" so a category absent during training does not crash transformation later. Its indicators become all zero; this allows processing but does not teach the model what the new category means.

Standard scaling subtracts the training mean and divides by the training standard deviation. It does not make a skewed distribution normal or remove outliers.

In [7]:
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("scikit-learn version:", sklearn.__version__)


def make_preprocessor(scale_numeric=True):
    """
    Create a fresh, unfitted preprocessor.

    Expected input:
        The 19-column X dataframe, with TotalCharges already
        converted to numeric using our fixed parsing rule.

    Learned values must later be fitted within training folds.
    """

    # Numerical preparation:
    # each training fold supplies its own replacement medians.
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    # Include scaling only when appropriate for the chosen model.
    if scale_numeric:
        numeric_steps.append(
            ("scaler", StandardScaler())
        )

    numeric_pipeline = Pipeline(steps=numeric_steps)

    # Preserve all recorded categories, including service-absence states.
    # Categories are learned when fit() is called on training data.
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop=None,
        sparse_output=False
    )

    # Apply the appropriate preparation to each feature group,
    # then combine the numerical and encoded categorical outputs.
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_encoder, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=True
    )


# Create separate objects so they do not share fitted state.
preprocessor_scaled = make_preprocessor(scale_numeric=True)
preprocessor_unscaled = make_preprocessor(scale_numeric=False)

print("Created scaled and unscaled preprocessing definitions.")
print("Neither preprocessor has been fitted.")

#ColumnTransformer applies different transformations to selected columns and combines their outputs.

scikit-learn version: 1.6.1
Created scaled and unscaled preprocessing definitions.
Neither preprocessor has been fitted.


A few choices in this code deserve explanation:
- No categorical imputation: your development checks found no missing categorical values. We are not inventing a replacement rule for a problem absent from the current data.
- drop=None: retains all category indicators. This is a starting representation; it creates redundant columns, so model-specific coefficient interpretation needs care.
- sparse_output=False: produces an ordinary numerical array, convenient for inspection and manageable for this small dataset.
- No feature removal: correlation alone did not justify dropping TotalCharges or another predictor.

**Preprocessing pipeline design**

We define separate scaled and unscaled preprocessing variants for model-specific use. Both apply training-fold median imputation to numerical features and one-hot encoding to categorical features. The scaled variant additionally standardises numerical features using training-fold means and standard deviations.

Meaningful service categories, including “No internet service” and “No phone service”, are retained. SeniorCitizen is treated as categorical despite its numerical storage. No categorical imputation is currently specified because the development checks found no missing categorical values.

All category indicators are initially retained. Unknown categories are encoded as all-zero indicators for the affected feature, allowing transformation without implying that their effects have been learned.

These are unfitted pipeline definitions. They will be combined with each estimator before cross-validation so learned preparation occurs within training folds. Their construction does not establish predictive performance.

Preprocessing verification

We temporarily divide the development data into training and validation portions to check the preprocessing code. Fresh diagnostic preprocessors are fitted only on the training portion and then applied to both portions.

We check that the transformations preserve customer counts, produce consistent feature columns, remove missing numerical values and use training-only imputation statistics.

This temporary division does not replace the saved development/test split or the planned five-fold model comparison. The reserved test set remains untouched. Diagnostic fitted preprocessors will not be reused for model evaluation.

In [8]:
import numpy as np
from sklearn.model_selection import train_test_split

# Temporary division of DEVELOPMENT data for technical checks only.
# The original development.csv and reserved test.csv are unchanged.
X_check_train, X_check_valid = train_test_split(
    X,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Confirm that the temporary portions contain different customers.
assert set(X_check_train.index).isdisjoint(X_check_valid.index)
assert len(X_check_train) + len(X_check_valid) == len(X)

print("Diagnostic training records:", len(X_check_train))
print("Diagnostic validation records:", len(X_check_valid))

# Calculate the expected medians from the training portion only.
# These are used to verify what the imputer learns.
expected_train_medians = (
    X_check_train[numeric_features].median().to_numpy()
)

check_results = []
diagnostic_preprocessors = {}

for variant, use_scaling in [
    ("Scaled", True),
    ("Unscaled", False),
]:
    # Create a fresh object for this diagnostic check.
    # The original preprocessing definitions remain unfitted.
    diagnostic = make_preprocessor(scale_numeric=use_scaling)

    # Learn replacement values, categories and (if enabled) scaling
    # from the diagnostic TRAINING portion only.
    train_transformed = diagnostic.fit_transform(X_check_train)

    # Apply those same learned rules to validation.
    # Do not call fit() or fit_transform() on validation.
    valid_transformed = diagnostic.transform(X_check_valid)

    feature_names = diagnostic.get_feature_names_out()

    # Transformation must preserve rows and produce matching columns.
    assert train_transformed.shape[0] == len(X_check_train)
    assert valid_transformed.shape[0] == len(X_check_valid)
    assert train_transformed.shape[1] == valid_transformed.shape[1]
    assert train_transformed.shape[1] == len(feature_names)

    # Outputs should contain usable numbers, with no NaN or infinity.
    assert np.isfinite(train_transformed).all()
    assert np.isfinite(valid_transformed).all()

    # Verify that imputation statistics match TRAINING medians.
    numeric_pipeline = diagnostic.named_transformers_["numeric"]
    learned_medians = numeric_pipeline.named_steps["imputer"].statistics_

    np.testing.assert_allclose(
        learned_medians,
        expected_train_medians
    )

    # In the scaled variant, numerical TRAINING columns should
    # have approximately zero mean after standardisation.
    if use_scaling:
        np.testing.assert_allclose(
            train_transformed[:, :len(numeric_features)].mean(axis=0),
            np.zeros(len(numeric_features)),
            atol=1e-10
        )

    diagnostic_preprocessors[variant] = diagnostic

    check_results.append({
        "Variant": variant,
        "Training rows": train_transformed.shape[0],
        "Validation rows": valid_transformed.shape[0],
        "Output features": train_transformed.shape[1],
        "No missing/infinite outputs": True,
        "Training-only medians verified": True,
    })

display(pd.DataFrame(check_results))

print("Training medians used in the diagnostic check:")
display(pd.Series(
    expected_train_medians,
    index=numeric_features,
    name="Training median"
))

print("All preprocessing checks passed.")

Diagnostic training records: 4507
Diagnostic validation records: 1127


,Variant,Training rows,Validation rows,Output features,No missing/infinite outputs,Training-only medians verified
0,Scaled,4507,1127,46,True,True
1,Unscaled,4507,1127,46,True,True


Training medians used in the diagnostic check:


,Training median
tenure,29.000
MonthlyCharges,70.350
TotalCharges,1377.975


All preprocessing checks passed.


what this checks:

| Check | Why it matters |
|---|---|
| Customer counts stay unchanged | Preprocessing must not silently remove records. |
| Training and validation have matching output columns | A model needs the same feature structure when making predictions. |
| No missing or infinite output values | Confirms the transformed data are numerically usable. |
| Imputer statistics equal training medians | Confirms the diagnostic imputer used the intended training data. |
| Scaled training features have approximately zero mean | Checks that numerical scaling operated correctly. |

Validation features do not need a mean of zero. They use the training mean and standard deviation; they do not calculate their own.

You will also see more than 19 output features because one-hot encoding expands categorical columns. This does not mean extra customers or extra source variables were added.

The 46 columns are 3 numerical columns plus 43 category indicators. The medians you printed belong only to this diagnostic training portion—do not hard-code them. Each cross-validation training fold will calculate its own.

Unseen-category handling check

We introduce an artificial, previously unseen payment-method value into a copy of one development validation record. This checks that the fitted encoder can transform an unfamiliar category without changing the output structure.

The original records remain unchanged. With handle_unknown="ignore", all indicators for the unfamiliar payment method should be zero.

This verifies software behaviour; it does not establish that predictions for unfamiliar categories will be accurate.

In [9]:
# Create a synthetic diagnostic example from one validation record.
# This copy does not change X, the validation data, or saved files.
unknown_example = X_check_valid.iloc[[0]].copy()
unseen_value = "__UNSEEN_PAYMENT_METHOD_CHECK__"

# Confirm the artificial value does not already occur in development.
assert unseen_value not in set(X["PaymentMethod"].dropna())

unknown_example["PaymentMethod"] = unseen_value

unknown_check_results = []

for variant, diagnostic in diagnostic_preprocessors.items():
    # Verify that the encoder did not learn this artificial category.
    encoder = diagnostic.named_transformers_["categorical"]
    payment_position = categorical_features.index("PaymentMethod")

    assert unseen_value not in encoder.categories_[payment_position]

    # Apply the already-fitted diagnostic preprocessor.
    # Do not refit it on the artificial example.
    transformed = diagnostic.transform(unknown_example)
    feature_names = diagnostic.get_feature_names_out()

    # Locate the one-hot columns belonging to PaymentMethod.
    payment_columns = [
        i
        for i, name in enumerate(feature_names)
        if name.startswith("categorical__PaymentMethod_")
    ]

    assert payment_columns, "PaymentMethod indicator columns not found."

    # The output structure must remain unchanged and numerically usable.
    assert transformed.shape == (1, len(feature_names))
    assert np.isfinite(transformed).all()

    # An unseen payment method should activate none of its known indicators.
    assert np.all(transformed[0, payment_columns] == 0)

    unknown_check_results.append({
        "Variant": variant,
        "Output features": transformed.shape[1],
        "Transformation succeeded": True,
        "Payment indicators all zero": True,
    })

display(pd.DataFrame(unknown_check_results))
print("Unseen-category checks passed.")

,Variant,Output features,Transformation succeeded,Payment indicators all zero
0,Scaled,46,True,True
1,Unscaled,46,True,True


Unseen-category checks passed.


**Initial feature retention, outlier and class-imbalance decisions**

We retain all 19 candidate predictors after excluding customerID and Churn. No additional value proxy, interaction feature or feature-selection step is included in the initial pipeline. The EDA findings do not independently establish that any remaining predictor should be removed.

We retain plausible numerical observations without outlier deletion, clipping or logarithmic transformation. Skewness and boxplot flags alone do not identify erroneous records. Strong correlations involving TotalCharges indicate overlapping information, but not complete redundancy.

We preserve the original class distribution in the initial preprocessing pipeline. Class weighting or resampling may be investigated during model development if justified. Any resampling will be confined to training folds, while validation and test data retain their original distributions.

These are initial methodological choices, not findings that the selected approach is optimal. Any later revision will be supported by development evidence and recorded with its reason. Predictive usefulness, interpretation and responsible use remain subject to evaluation.

Reusable preprocessing handover

The shared preprocessing module contains the predictor lists, fixed TotalCharges parsing rule and a function that creates a fresh preprocessing pipeline.

The pipeline expects the 19 original predictor columns, excluding customerID and Churn. It converts TotalCharges to numeric, imputes numerical missing values, optionally scales numerical features and one-hot encodes categorical features.

Fixed parsing is included in the pipeline so subsequent members apply the same rule consistently. Learned transformations remain unfitted until the complete model pipeline is fitted on training data. Diagnostic fitted objects are not used for model comparison.

In [10]:
# File created by this cell: src/telco_preprocessing.py
#It will stop rather than overwrite a different existing version.

# Store reusable definitions in the shared project.
SRC_DIR = PROJECT_DIR / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)

MODULE_PATH = SRC_DIR / "telco_preprocessing.py"

module_code = '''
"""Shared preprocessing definitions for Group 2026-DS-77."""

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)

NUMERIC_FEATURES = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

CATEGORICAL_FEATURES = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

PREDICTOR_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def parse_predictors(frame):
    """Validate predictor columns and apply fixed numeric parsing.

    This function learns no statistics and never modifies its input.
    Missing TotalCharges values remain missing for later imputation.
    """
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("Expected a pandas DataFrame of predictor columns.")

    if not frame.columns.is_unique:
        raise ValueError("Duplicate column names found.")

    missing = set(PREDICTOR_COLUMNS) - set(frame.columns)
    unexpected = set(frame.columns) - set(PREDICTOR_COLUMNS)

    if missing or unexpected:
        raise ValueError(
            f"Predictor schema mismatch. "
            f"Missing: {sorted(missing)}; "
            f"Unexpected: {sorted(unexpected)}"
        )

    # Select a consistent column order and copy before making changes.
    result = frame.loc[:, PREDICTOR_COLUMNS].copy()

    text = result["TotalCharges"].astype("string").str.strip()
    blank = text.isna() | text.eq("").fillna(False)

    numeric = pd.to_numeric(text.mask(blank), errors="coerce")
    unexpected_text = numeric.isna() & ~blank

    if unexpected_text.any():
        raise ValueError("Unexpected non-numeric TotalCharges values found.")

    result["TotalCharges"] = numeric.astype("float64")

    # Our current categorical policy assumes complete values.
    # Stop for review if a later input violates that assumption.
    categories = result[CATEGORICAL_FEATURES]
    if categories.isna().any().any():
        raise ValueError("Missing categorical values require review.")

    for column in CATEGORICAL_FEATURES:
        if result[column].astype("string").str.strip().eq("").any():
            raise ValueError(f"Blank categorical value in {column}.")

    return result


def make_model_preprocessor(scale_numeric=True):
    """Return a NEW, UNFITTED pipeline for the 19 raw predictors."""

    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]

    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    columns = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(numeric_steps),
                NUMERIC_FEATURES,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop=None,
                    sparse_output=False,
                ),
                CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )

    return Pipeline([
        (
            "parse",
            FunctionTransformer(parse_predictors, validate=False),
        ),
        ("columns", columns),
    ])
'''

if MODULE_PATH.exists():
    existing_code = MODULE_PATH.read_text(encoding="utf-8")
    if existing_code != module_code:
        raise FileExistsError(
            f"A different module already exists: {MODULE_PATH}. "
            "Review it before replacing it."
        )
    print("Reusing the identical shared module.")
else:
    MODULE_PATH.write_text(module_code, encoding="utf-8")
    print("Created shared module:", MODULE_PATH)

#Why check the column names? If someone accidentally includes Churn or customerID, the module stops instead of silently accepting an incorrect input.
#The categorical checks also make our current assumption explicit: unfamiliar categories are supported, but missing categorical values require a reviewed treatment.

Reusing the identical shared module.


In [11]:
#Verify that the shared module reproduces your existing results
#We changed how the code is packaged, so we should verify that its transformations still match the ones already checked.

import importlib
import sys

# Make the shared src folder available for Python imports.
src_location = str(SRC_DIR)
if src_location not in sys.path:
    sys.path.insert(0, src_location)

importlib.invalidate_caches()
import telco_preprocessing as telco_prep
telco_prep = importlib.reload(telco_prep)

# Confirm that Python imported this project's file.
assert Path(telco_prep.__file__).resolve() == MODULE_PATH.resolve()

# Use the original predictor values from development.
# TotalCharges parsing now happens INSIDE the shared pipeline.
X_raw = dev.drop(columns=["customerID", "Churn"]).copy()
X_raw_before = X_raw.copy(deep=True)

# Reuse exactly the same diagnostic row membership as before.
raw_check_train = X_raw.loc[X_check_train.index]
raw_check_valid = X_raw.loc[X_check_valid.index]

module_check_results = []

for variant, use_scaling in [
    ("Scaled", True),
    ("Unscaled", False),
]:
    # Fresh diagnostic instance; do not reuse it for model evaluation.
    shared_check = telco_prep.make_model_preprocessor(
        scale_numeric=use_scaling
    )

    shared_check.fit(raw_check_train)
    shared_output = shared_check.transform(raw_check_valid)

    # Previously checked version used already-parsed input.
    previous_output = diagnostic_preprocessors[variant].transform(
        X_check_valid
    )

    # Packaging the same rules should reproduce the same results.
    np.testing.assert_allclose(
        shared_output,
        previous_output,
        rtol=1e-10,
        atol=1e-10,
    )

    module_check_results.append({
        "Variant": variant,
        "Validation rows": shared_output.shape[0],
        "Output features": shared_output.shape[1],
        "Matches previous pipeline": True,
    })

# Confirm that preprocessing did not modify the original input.
pd.testing.assert_frame_equal(X_raw, X_raw_before)

display(pd.DataFrame(module_check_results))
print("Shared module checks passed.")
print("Original predictor data unchanged.")

# Remove the last fitted diagnostic instance to avoid accidental reuse.
del shared_check

,Variant,Validation rows,Output features,Matches previous pipeline
0,Scaled,1127,46,True
1,Unscaled,1127,46,True


Shared module checks passed.
Original predictor data unchanged.


What Member 4 will receive

Member 4 will import make_model_preprocessor() and place a fresh instance inside the same pipeline as each classifier.

Cross-validation will then fit that complete pipeline on each training fold.

They should not receive a single preprocessed development CSV for cross-validation, because its learned preparation could have used the validation records.

Member 3 now has reusable preprocessing code.

In [12]:
#Save the check results

import hashlib
import json
import platform
from datetime import datetime
from zoneinfo import ZoneInfo

TABLE_DIR = PROJECT_DIR / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Save the actual results collected by the earlier verification cells.
verification_tables = {
    "preprocessing_missingness.csv": missing_check,
    "preprocessing_diagnostic_checks.csv": pd.DataFrame(check_results),
    "preprocessing_unknown_category_checks.csv": pd.DataFrame(
        unknown_check_results
    ),
    "preprocessing_module_checks.csv": pd.DataFrame(module_check_results),
}

for filename, table in verification_tables.items():
    # The missingness table uses feature names as its index.
    save_index = filename == "preprocessing_missingness.csv"
    table.to_csv(TABLE_DIR / filename, index=save_index)

# Identify the exact shared code and development file used in this run.
# A hash identifies file contents; it does not establish scientific validity.
module_hash = hashlib.sha256(MODULE_PATH.read_bytes()).hexdigest()
development_hash = hashlib.sha256(DEV_PATH.read_bytes()).hexdigest()

run_record = {
    "recorded_at": datetime.now(
        ZoneInfo("Asia/Colombo")
    ).isoformat(),
    "responsible_member": "IT24103676",
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "development_file": str(DEV_PATH),
    "development_sha256": development_hash,
    "development_records": len(dev),
    "source_predictor_count": X.shape[1],
    "module_file": str(MODULE_PATH),
    "module_sha256": module_hash,
    "diagnostic_split": {
        "source": "development data only",
        "validation_fraction": 0.20,
        "random_state": 42,
        "stratified": True,
        "training_records": len(X_check_train),
        "validation_records": len(X_check_valid),
    },
    "verification_tables": list(verification_tables),
    "scope": (
        "Preprocessing technical checks only; "
        "predictive performance has not been evaluated."
    ),
}

RUN_RECORD_PATH = TABLE_DIR / "preprocessing_run_record.json"
RUN_RECORD_PATH.write_text(
    json.dumps(run_record, indent=2),
    encoding="utf-8"
)

# Verify that every expected export exists and contains data.
expected_exports = [
    TABLE_DIR / filename for filename in verification_tables
] + [RUN_RECORD_PATH]

for path in expected_exports:
    assert path.is_file(), f"Missing export: {path}"
    assert path.stat().st_size > 0, f"Empty export: {path}"
    print("Saved:", path.name)

print("Preprocessing evidence exports completed.")

#These files preserve what was checked and which version of the code was used. The transformed customer data and diagnostic fitted objects do not need to be saved.

Saved: preprocessing_missingness.csv
Saved: preprocessing_diagnostic_checks.csv
Saved: preprocessing_unknown_category_checks.csv
Saved: preprocessing_module_checks.csv
Saved: preprocessing_run_record.json
Preprocessing evidence exports completed.


**Member 3 handover — preprocessing and feature engineering**

**Responsible member:** IT24103676  
**Receiving member:** IT24102789, with group-leader review by IT24103677.

**Completed work**

- Loaded the existing development subset and confirmed 5,634 records, 19 candidate predictors and the expected target counts.
- Separated customerID and Churn from model inputs; encoded Churn as No = 0 and Yes = 1.
- Preserved eight missing TotalCharges values during fixed numerical parsing.
- Defined training-fold median imputation and optional numerical standardisation.
- Defined one-hot encoding for 16 categorical predictors, preserving service-absence categories.
- Retained all 19 candidate predictors and plausible numerical observations in the initial approach.
- Left resampling out of the initial preprocessing pipeline.
- Created the reusable module at `src/telco_preprocessing.py`.

**Verification evidence**

Both scaled and unscaled variants passed diagnostic checks using 4,507 development training records and 1,127 development validation records. Each produced 46 output features. Checks covered row preservation, consistent output dimensions, finite outputs, training-only imputation statistics and numerical centring where applicable.

An artificial unseen payment-method category was transformed successfully, with all corresponding payment-method indicators set to zero. The shared module reproduced the earlier diagnostic transformations without modifying the original predictor data.

These checks demonstrate technical behaviour, not predictive effectiveness. The diagnostic output size of 46 is an observation from this training subset; category vocabulary is learned separately during each model fit.

**Instructions for Member 4**

1. Reuse the existing development/test split.
2. Load the original development predictors, excluding customerID and Churn.
3. Import `make_model_preprocessor` from the shared module.
4. Create a fresh preprocessor and combine it with the classifier in one model pipeline.
5. Pass the complete pipeline into cross-validation so all learned preprocessing is fitted within each training fold.
6. Select scaled or unscaled numerical preparation with a model-specific justification.
7. Do not reuse diagnostic fitted preprocessors or hard-code the diagnostic medians.
8. Record any changes to preprocessing and update the shared module and logs together.

**Remaining responsibilities**

Member 4 must implement the agreed model-comparison protocol and investigate justified alternatives. The group leader must review the protocol and later evaluate the frozen procedure on the reserved test set.

Fresh-session top-to-bottom execution and verification-export inspection were completed on 24/09/2026, as recorded below. Group-leader review and Member 4’s handover acceptance should be recorded when they occur.

**Limitations**

Median imputation does not recover the true missing charges. Unseen-category handling does not establish prediction accuracy for unfamiliar categories. Correlated predictors can complicate interpretation. Dataset timing and representativeness limitations remain unchanged.

Fresh-session top-to-bottom execution completed successfully on 12/10/2026. Verification exports inspected. Shared module ready for Member 4’s integration.